## Imports

In [11]:
import os
from typing import Any

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

## Constants

In [3]:
def seed_everything(seed: int = 334791) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()

In [14]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

user_column = "user_id"
item_column = "item_id"
event_type_column = "event_type"
watch_time_column = "watch_time"
date_column = "date"

device = get_device()

MAX_SEQ_LEN = 200
NUM_NEGS = 512
TARGET_WATCH_TIME_MIN = np.log(60)

device

device(type='cuda')

## Data

In [15]:
def split_data(data: pd.DataFrame, column: str, splt_value: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_data = data[data[column].dt.date < splt_value]
    test_data = data[data[column].dt.date  >= splt_value]
    return train_data, test_data

dataset = pl.read_parquet("data/train.parquet").to_pandas()

dataset[watch_time_column] = np.log(dataset[watch_time_column] + 1)

train, test = split_data(dataset, date_column, pd.to_datetime("2024-12-02").date())
train, val = split_data(train, date_column, pd.to_datetime("2024-12-01").date())

target_users = pl.read_parquet("data/target_user_ids.parquet").to_pandas()
target_user_ids = target_users["user_id"].values

## Vocab

In [13]:
def build_vocab(data: pd.DataFrame, num_bins: int) -> tuple[dict[int, int], dict[int, int], dict[int, int], np.ndarray]:
    items = sorted(train[item_column].unique())
    item_to_idx = {item: i + 1 for i, item in enumerate(items)}
    idx_to_item = {v: k for k, v in item_to_idx.items()}

    event_types  = ["watch_time", "like", "favorite"]
    event_to_idx = {e: i + 1 for i, e in enumerate(event_types)}

    wt_values = train.loc[train[event_type_column] == "watch_time", "watch_time"].values
    edges = np.quantile(wt_values, np.linspace(0, 1, num_bins + 1))
    edges[0]  = -np.inf
    edges[-1] =  np.inf

    return item_to_idx, idx_to_item, event_to_idx, edges

item_to_idx, idx_to_item, event_to_idx, wt_bins = build_vocab(train, 14)
num_items = len(item_to_idx)
num_event_types = len(event_to_idx)

num_items, num_event_types

(1651686, 3)

## Dataset to sequence

In [28]:
def encode_sequences(data: pd.DataFrame, item_to_idx: dict[int, int], event_to_idx: dict[int, int], wt_bins: np.ndarray) -> dict[int, list[tuple[int, int, int]]]:
    data = data.sort_values([user_column, date_column]).copy()

    data["item_idx"]  = data[item_column].map(item_to_idx).fillna(0).astype(int)
    data["event_idx"] = data[event_type_column].map(event_to_idx).astype(int)

    wt_bin = np.zeros(len(data), dtype=int)
    wt_mask = (data[event_type_column] == "watch_time").values
    wt_bin[wt_mask] = (
        np.searchsorted(wt_bins[1:-1], data.loc[wt_mask, watch_time_column].values) + 1
    ).astype(int)
    data["wt_bin"] = wt_bin

    data["is_target"] = (
        data[event_type_column].isin(["like", "favorite"])
        | ((data[event_type_column] == "watch_time") & (data[watch_time_column] > TARGET_WATCH_TIME_MIN))
    )

    user_seqs = {}
    for user_id, features in tqdm(data.groupby(user_column, sort=False)):
        user_seqs[user_id] = list(zip(
            features["item_idx"].tolist(),
            features["event_idx"].tolist(),
            features["wt_bin"].tolist(),
            features["is_target"].tolist(),
        ))
    return user_seqs

In [29]:
train_seqs = encode_sequences(train, item_to_idx, event_to_idx, wt_bins)
len(train_seqs), train_seqs[list(train_seqs.keys())[0]][:10]

100%|██████████| 400196/400196 [01:09<00:00, 5792.84it/s]


(400196,
 [(275164, 3, 0, True),
  (27104, 3, 0, True),
  (199894, 3, 0, True),
  (690812, 3, 0, True),
  (149100, 3, 0, True),
  (156090, 3, 0, True),
  (936697, 3, 0, True),
  (147988, 3, 0, True),
  (408177, 3, 0, True),
  (307483, 3, 0, True)])

## Dataset

In [30]:
class Dataset(TorchDataset):
    def __init__(self, user_seqs: dict[int, list[tuple[int, int, int]]], num_items: int, max_seq_len: int, num_negs: int) -> None:
        self.num_items = num_items
        self.max_seq_len = max_seq_len
        self.num_negs = num_negs

        self.examples = []
        for seq in tqdm(user_seqs.values()):
            if len(seq) < 3:
                continue
            if len(seq) > max_seq_len + 1:
                seq = seq[-(max_seq_len + 1):]
            for t in range(1, len(seq)):
                if not seq[t][3]:
                    continue
                prefix = seq[:t]
                target = seq[t][0] 
                is_target = seq[t][3]   
                self.examples.append((prefix, target, is_target))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        prefix, target, is_target = self.examples[idx]

        neg = np.random.randint(1, self.num_items + 1, size=self.num_negs)
        collision = (neg == target)
        if collision.any():
            neg[collision] = np.random.randint(1, self.num_items + 1, size=collision.sum())

        return {
            "inp_items": [s[0] for s in prefix], 
            "inp_events": [s[1] for s in prefix],
            "inp_wt": [s[2] for s in prefix],
            "target": target,
            "neg": neg,
            "is_target":  int(is_target),
        }

In [31]:
train_dataset = Dataset(train_seqs, num_items, MAX_SEQ_LEN, NUM_NEGS)

100%|██████████| 400196/400196 [00:50<00:00, 7924.42it/s] 


## Collate fn

In [32]:
def collate_fn(batch: list[torch.Tensor]) -> dict[str, torch.Tensor]:
    max_len = max(len(b["inp_items"]) for b in batch) + 1

    inp_items, inp_events, inp_wt, targets, is_targets, negs = [], [], [], [], [], []
    for b in batch:
        items = b["inp_items"] + [num_items + 1]
        events = b["inp_events"] + [0]
        wt = b["inp_wt"] + [0]
        pad = max_len - len(items)
        inp_items.append([0] * pad + items)
        inp_events.append([0] * pad + events)
        inp_wt.append([0] * pad + wt)
        targets.append(b["target"])
        is_targets.append(b["is_target"])
        negs.append(b["neg"])

    return {
        "inp_items": torch.tensor(inp_items, dtype=torch.long),
        "inp_events": torch.tensor(inp_events, dtype=torch.long),
        "inp_wt": torch.tensor(inp_wt, dtype=torch.long),
        "target": torch.tensor(targets, dtype=torch.long),
        "is_target":  torch.tensor(is_targets,  dtype=torch.float),
        "neg": torch.tensor(np.stack(negs), dtype=torch.long),
    }

In [33]:
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, num_workers=0, collate_fn=collate_fn)

In [35]:
%%time
batch = next(iter(train_loader))
batch["inp_items"]

CPU times: user 948 ms, sys: 306 ms, total: 1.25 s
Wall time: 1.13 s


tensor([[      0,       0,       0,  ...,   68164,  115549, 1651687],
        [      0,       0,       0,  ...,  302818,  283595, 1651687],
        [      0,       0,       0,  ...,  274796,  179194, 1651687],
        ...,
        [      0,       0,       0,  ..., 1172437,  179110, 1651687],
        [      0,       0,       0,  ...,  283483,  432966, 1651687],
        [      0,       0,       0,  ...,   31016,   67468, 1651687]])

## Модель

In [40]:
class gSASRec(nn.Module):

    def __init__(
        self,
        num_items: int,
        num_event_types: int,
        num_wt_bins: int,
        d_model: int,
        num_heads: int,
        num_blocks: int,
        dropout: float,
        max_seq_len: int
    ):
        super().__init__()
        self.query_idx = num_items + 1
        self.item_emb  = nn.Embedding(num_items + 2, d_model, padding_idx=0)
        self.event_emb = nn.Embedding(num_event_types + 1, d_model, padding_idx=0)
        self.wt_emb    = nn.Embedding(num_wt_bins, d_model, padding_idx=0)
        self.pos_emb   = nn.Embedding(max_seq_len + 2, d_model)

        self.input_ln = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_blocks)
        self.max_seq_len = max_seq_len

    def forward(self, inp_items: torch.Tensor, inp_events: torch.Tensor, inp_wt: torch.Tensor) -> torch.Tensor:
        B, L = inp_items.shape

        pos = torch.arange(L, 0, -1, device=inp_items.device).unsqueeze(0)

        x = (
            self.item_emb(inp_items)
            + self.event_emb(inp_events)
            + self.wt_emb(inp_wt)
            + self.pos_emb(pos)
        )
        x = self.input_ln(x)
        x = self.drop(x)

        causal_mask = torch.triu(
            torch.full((L, L), float('-inf'), device=x.device), diagonal=1
        )
        hidden = self.transformer(x, mask=causal_mask)

        pad_token_mask = (inp_items == 0).unsqueeze(-1)
        hidden = hidden.masked_fill(pad_token_mask, 0.0)

        return hidden  


    def item_emb_norm(self, idx):
        e = self.item_emb(idx)
        return F.normalize(e, dim=-1)

In [41]:
model = gSASRec(
    num_items=num_items,
    num_event_types=num_event_types,
    num_wt_bins=len(wt_bins),
    d_model=64,
    num_heads=2,
    num_blocks=2,
    dropout=0.1,
    max_seq_len=MAX_SEQ_LEN,
).to(device)

/tmp/ipykernel_269/987287066.py:32: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=num_blocks)


In [42]:
with torch.no_grad():
    out = model(
        batch["inp_items"][:2].to(device),
        batch["inp_events"][:2].to(device),
        batch["inp_wt"][:2].to(device),
    )
out.shape

torch.Size([2, 201, 64])

## Training

In [43]:
def sampled_softmax_loss(last_hidden: torch.Tensor, target: torch.Tensor, neg: torch.Tensor, model: gSASRec) -> torch.Tensor:
    h = F.normalize(last_hidden, dim=-1)
    
    pos_emb = model.item_emb_norm(target) 
    pos_score = (h * pos_emb).sum(-1, keepdim=True)

    neg_emb = model.item_emb_norm(neg)    
    neg_score = (h.unsqueeze(1) * neg_emb).sum(-1) 

    logits = torch.cat([pos_score, neg_score], dim=-1)     
    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=last_hidden.device)  

    return F.cross_entropy(logits, labels)
    

def gbce_loss(last_hidden: torch.Tensor, target: torch.Tensor, neg: torch.Tensor, model: gSASRec) -> torch.Tensor:
    h = F.normalize(last_hidden, dim=-1)     

    pos_emb = model.item_emb_norm(target)   
    pos_score = (h * pos_emb).sum(-1)        

    neg_emb = model.item_emb_norm(neg)      
    neg_score = (h.unsqueeze(1) * neg_emb).sum(-1) 

    pos_loss = -F.logsigmoid(pos_score)             
    neg_loss = -F.logsigmoid(-neg_score).sum(-1) / neg.shape[1]

    return (pos_loss + neg_loss).mean()

In [44]:
sampled_softmax_loss(
    out[:2, -1, :],
    batch["target"][:2].to(device),
    batch["neg"][:2].to(device),
    model,
)

# gbce_loss(
#     out[:2, -1, :],
#     batch["target"][:2].to(device),
#     batch["neg"][:2].to(device),
#     model,
# )

tensor(6.1847, device='cuda:0', grad_fn=<NllLossBackward0>)

In [129]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

val_interacted_items = get_interacted_items(val, user_column, item_column)
train_interacted_items = get_interacted_items(train, user_column, item_column)
val_users_in_train = {u: s for u, s in train_seqs.items() if u in val_interacted_items}

global_step = 0

In [130]:
def train_epoch(model: gSASRec, loader: DataLoader, optimizer: torch.optim.Adam, device: torch.device, global_step=0):
    model.train()
    total_loss, n = 0.0, 0
    dataset_len = len(loader)
    batch_loss = 0.0
    for batch in tqdm(loader):
        inp_items = batch["inp_items"].to(device)
        inp_events = batch["inp_events"].to(device)
        inp_wt = batch["inp_wt"].to(device)
        target = batch["target"].to(device)
        neg = batch["neg"].to(device)

        hidden = model(inp_items, inp_events, inp_wt)
        last_hidden = hidden[:, -1, :]
        
        loss = sampled_softmax_loss(last_hidden, target, neg, model)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        batch_loss += loss.item()
        n += 1

        if n % 5000 == 0:
            print(f"Loss: {batch_loss / 5000}")
            batch_loss = 0.0

        global_step += 1

    return total_loss / n, global_step


@torch.no_grad()
def recommend_all(
    model: gSASRec,
    user_seqs: dict[int, list[int]],
    seen_items: dict[int, list[int]],
    item_to_idx: dict[int, int],
    idx_to_item: dict[int, int],
    device: torch.device,
    k: int = 20,
    batch_size: int = 512,
):
    model.eval()
    n_items  = model.item_emb.num_embeddings - 2 
    all_idx  = torch.arange(1, n_items + 1, device=device)
    all_emb  = model.item_emb_norm(all_idx)

    seen_idxs_map = {}
    for user_id, items in seen_items.items():
        idxs = [item_to_idx[iid] - 1 for iid in items if iid in item_to_idx]
        if idxs:
            seen_idxs_map[user_id] = idxs

    recs  = {}
    users = list(user_seqs.keys())

    for start in range(0, len(users), batch_size):
        batch_users = users[start : start + batch_size]
        batch_seqs = [user_seqs[u][-MAX_SEQ_LEN:] for u in batch_users]
        max_len = max(len(s) for s in batch_seqs) + 1

        all_items, all_events, all_watch = [], [], []
        for seq in batch_seqs:
            items = [s[0] for s in seq] + [model.query_idx]
            all_events = [s[1] for s in seq] + [0]
            wt = [s[2] for s in seq] + [0]
            pad = max_len - len(items)
            all_items.append([0] * pad + items)
            all_events.append([0] * pad + events)
            all_watch.append([0] * pad + wt)

        inp_items = torch.tensor(all_items, dtype=torch.long, device=device)
        inp_events = torch.tensor(all_events, dtype=torch.long, device=device)
        inp_wt = torch.tensor(all_watch, dtype=torch.long, device=device)

        hidden = model(inp_items, inp_events, inp_wt)

        h = hidden[:, -1, :]
        last_hidden = F.normalize(h, dim=-1)     
        
        scores = last_hidden @ all_emb.T

        for i, user_id in enumerate(batch_users):
            idxs = seen_idxs_map.get(user_id)
            if idxs:
                scores[i, idxs] = -1e9

        top_k_all = torch.topk(scores, k, dim=1).indices.cpu()
        for i, user_id in enumerate(batch_users):
            recs[user_id] = [
                idx_to_item[int(idx) + 1]
                for idx in top_k_all[i]
                if (int(idx) + 1) in idx_to_item
            ]

    return recs

In [ ]:
metrics, losses = [], []

for epoch in range(1, 30 + 1):
    loss, global_step = train_epoch(
        model, train_loader, optimizer, device, global_step=global_step
    )

    recs = recommend_all(
        model, val_users_in_train, train_interacted_items, item_to_idx, idx_to_item, device, k=30
    )
    p20 = get_precision_at_k(recs, val_interacted_items, 20)
    metrics.append(p20)
    losses.append(loss)
    
    print(f"  Epoch {epoch} | loss={loss} | val precision@20={p20}")

    with open(f"data/metrics_{epoch}.json", "w") as f:
        json.dump({"metrics": metrics, "loss": losses}, f)